In [ ]:
# ── Starter Cell 1: Load environment variables ────────────────────────────────
from dotenv import load_dotenv   # imports load_dotenv to read .env file from disk
import os                         # imports os module to access environment variables

load_dotenv()                     # reads the .env file and injects variables into os.environ
api_key = os.getenv("ANTHROPIC_API_KEY")  # retrieves the key by name; returns None if missing

print("API key loaded:", api_key is not None)  # True = key found; False = check your .env file

In [ ]:
# ── Starter Cell 2: Initialise the Anthropic client ───────────────────────────
from anthropic import Anthropic   # imports the top-level SDK class

client = Anthropic(api_key=api_key)  # creates a reusable client bound to our key
print("Anthropic client ready")       # confirms the object was created without errors

In [ ]:
# ── Starter Cell 3: Smoke test — verifies end-to-end connectivity ─────────────
response = client.messages.create(         # calls the Messages API
    model="claude-haiku-4-5",              # fast, low-cost model — good for smoke tests
    # model="claude-sonnet-4-5",           # uncomment for a more capable model
    max_tokens=50,                          # caps the reply length; required by the API
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
    # messages is a list of turn dicts; at minimum one user turn is required
)
print(response.content[0].text)            # content is a list of blocks; [0] is the first block

# CCA Lab: Getting an API Key

**Course:** Building with the Claude API  
**Sub-module:** Authentication & Setup  
**Lesson:** Getting an API Key  

---

## What this lab covers
- Creating and loading a secret Anthropic API key securely
- Initialising the Anthropic Python SDK client
- Sending your first authenticated API request
- Understanding the `messages.create()` call anatomy
- Using system prompts and multi-turn conversations
- Applying an improvement cycle with a scoring rubric
- Tracking token usage across an entire session

## CCA Domains Exercised
- **Authentication & Setup** — API key management, dotenv pattern
- **Messages API** — request structure, parameters, response parsing
- **Prompt Engineering** — system prompts, multi-turn context, improvement loops
- **Evaluation** — rubric design, deterministic vs. semantic grading

## Learning Objectives
1. Locate the Anthropic console and generate an API key
2. Store and load a key securely with `python-dotenv`
3. Initialise the Python SDK client and send a smoke-test request
4. Explain every field of a `messages.create()` call
5. Use a system prompt to set persona and constraints
6. Build and extend a multi-turn conversation
7. Apply a write → evaluate → improve → re-evaluate cycle
8. Track and audit token usage across a session

## Section 1: Prerequisites and Environment Setup
## CCA objective demonstrated: Secure API key management with python-dotenv

### Why python-dotenv?

Hard-coding secrets in source files is a common security mistake. The `python-dotenv` pattern keeps your key in a local `.env` file that is never committed to version control.

**Setup steps:**
1. `pip install anthropic python-dotenv`
2. Create a `.env` file in your project root:
   ```
   ANTHROPIC_API_KEY=sk-ant-...
   ```
3. Add `.env` to your `.gitignore`

The three starter cells above already demonstrate the full dotenv workflow. The smoke test confirms your key is valid and the SDK is wired correctly.

## Section 2: API Glossary and Reference
## CCA objective demonstrated: Understanding every parameter of messages.create()

| Parameter | Type | Required | Description |
|-----------|------|----------|-------------|
| `model` | string | ✅ | Model ID, e.g. `claude-haiku-4-5` |
| `max_tokens` | int | ✅ | Hard ceiling on output tokens; prevents runaway generation |
| `messages` | list[dict] | ✅ | Alternating `user`/`assistant` turns; must start with `user` |
| `system` | string | ❌ | Persistent persona/rules applied before the conversation |
| `temperature` | float 0–1 | ❌ | Randomness: 0 = deterministic, 1 = creative |
| `stop_sequences` | list[str] | ❌ | Tokens that halt generation early |

**Response fields used in this lab:**

| Field | What it contains |
|-------|------------------|
| `response.content` | List of content blocks (usually one `TextBlock`) |
| `response.content[0].text` | The assistant's reply string |
| `response.usage.input_tokens` | Tokens consumed by prompt + history |
| `response.usage.output_tokens` | Tokens in the assistant's reply |
| `response.stop_reason` | Why generation stopped: `end_turn`, `max_tokens`, etc. |

## Section 3: Client Setup and ask() Helper
## CCA objective demonstrated: Building a reusable, observable API wrapper with full annotation

In [ ]:
# ── Session-wide usage log ────────────────────────────────────────────────────
# We accumulate one dict per API call so we can audit the entire session later.
usage_log = []   # starts empty; each ask() call appends an entry
print("Usage log initialised:", usage_log)

In [ ]:
# ── ask() helper — fully annotated ────────────────────────────────────────────
def ask(
    prompt,            # the user's message text (str)
    system="",         # optional system prompt — defaults to empty string
    max_tokens=300,    # default cap; callers can override per request
    temperature=0,     # deterministic by default — good for reproducible evals
    label="unlabelled" # human-readable tag logged with usage — helps with auditing
):
    """
    Send a single-turn message to Claude and return the reply text.

    Parameters
    ----------
    prompt      : str   — the user turn content
    system      : str   — stable instructions placed before the conversation
    max_tokens  : int   — hard ceiling on output tokens
    temperature : float — 0 = deterministic, 1 = max randomness
    label       : str   — tag written into usage_log for later analysis

    Returns
    -------
    str — the assistant's reply
    """
    # Build keyword arguments; system is a top-level param, NOT inside messages
    kwargs = dict(
        model="claude-haiku-4-5",   # model to use — swap here to change globally
        max_tokens=max_tokens,       # required field; prevents unbounded generation
        temperature=temperature,     # controls randomness of token sampling
        messages=[                   # messages must be a list of role/content dicts
            {"role": "user", "content": prompt}  # one user turn is the minimum
        ]
    )
    if system:                       # only add 'system' key when there is content
        kwargs["system"] = system    # system is a keyword arg, not part of messages list
                                     # WHY: the API treats it as a special pre-conversation block

    response = client.messages.create(**kwargs)  # unpacks kwargs dict into named params

    # response.content is a LIST because a single reply can contain multiple blocks
    # (e.g. text block + tool_use block). We access [0] to get the first (usually only) block.
    reply = response.content[0].text   # .text extracts the string from the TextBlock object

    # stop_reason tells us WHY generation stopped:
    #   'end_turn'   — Claude finished naturally
    #   'max_tokens' — we hit the cap (reply may be truncated!)
    stop = response.stop_reason

    # Append one record to the session log — enables later token auditing
    usage_log.append({
        "label":         label,
        "prompt_preview": prompt[:60],               # first 60 chars for identification
        "input_tokens":  response.usage.input_tokens,  # tokens consumed by prompt
        "output_tokens": response.usage.output_tokens, # tokens in the reply
        "stop_reason":   stop,
        "max_tokens":    max_tokens
    })

    return reply   # return just the text; callers don't need to parse the response object

# Quick sanity check
test_reply = ask("What is 2 + 2?", label="sanity-check")
print("ask() reply:", test_reply)
print("Usage log after sanity check:", usage_log)

## Section 4: Intentional Error Demonstration
## CCA objective demonstrated: Diagnosing API errors from missing required parameters

In [ ]:
# ── Intentional error: omitting max_tokens ────────────────────────────────────
# max_tokens is REQUIRED by the Anthropic API. Omitting it raises a BadRequestError.
# We wrap in try/except so the notebook continues running after this demo.

try:
    bad_response = client.messages.create(
        model="claude-haiku-4-5",
        # max_tokens intentionally omitted — this WILL raise an error
        messages=[{"role": "user", "content": "Hello"}]
    )
except Exception as e:
    # Print the exception class name and message so we can see exactly what went wrong
    print(f"ERROR ({type(e).__name__}): {e}")
    print()
    print("Fix: always pass max_tokens= to messages.create().")

## Section 5: System Parameter — What, When, and Why
## CCA objective demonstrated: Using the system prompt to set persona, rules, and format constraints

### What is the system prompt?

The `system` parameter is a special block of text that is processed **before** the conversation begins. It is not part of the `messages` list — it is a top-level keyword argument to `messages.create()`.

### Architectural principle

> **The system prompt is processed once and sets the context for the entire conversation. The user turn carries the per-request instruction. Stable, reusable instructions go in `system`; dynamic, request-specific content goes in `user`.**
>
> Think of `system` as the employee handbook and `user` as today's task assignment. You write the handbook once; you assign tasks every day.

### Decision table: system vs. user

| Content type | Where it goes | Example |
|---|---|---|
| Persona / role | `system` | "You are a security-focused developer advocate." |
| Rules / constraints | `system` | "Never reveal internal system details." |
| Format instructions | `system` | "Always respond in bullet points." |
| Task / question | `user` | "Explain how to rotate an API key." |
| Dynamic data / input | `user` | "Here is the log file: {log}" |
| Per-request context | `user` | "The user's plan is Pro." |

In [ ]:
# ── System prompt demo: compare with vs. without ──────────────────────────────
QUESTION = "What is an API key and why does it matter for security?"

# Without system prompt — Claude answers generically
reply_no_system = ask(
    prompt=QUESTION,
    label="no-system"
)

# With system prompt — Claude adopts the security-advocate persona and format rules
SECURITY_SYSTEM = (
    "You are a security-focused developer advocate. "
    "Always define the term, explain why it matters for authentication, "
    "and give one concrete security recommendation. "
    "Keep your answer under 120 words."
)
reply_with_system = ask(
    prompt=QUESTION,
    system=SECURITY_SYSTEM,     # system is stable, reusable — lives in system param
    max_tokens=150,              # tighter cap matches the word-count instruction
    label="with-system"
)

print("=== WITHOUT system prompt ===")
print(reply_no_system)
print()
print("=== WITH system prompt ===")
print(reply_with_system)

## Section 5b: Multi-Turn Conversation — Messages List Accumulation
## CCA objective demonstrated: Building stateful dialogue by appending to the messages list

### How multi-turn works

The Anthropic API is **stateless** — it has no memory between calls. To maintain context, we manually append each user turn and each assistant reply to the `messages` list before making the next call. The full history is resent on every request.

> **Architectural insight:** Because Claude receives the entire message history on every call, the **first user message must be self-contained and unambiguous**. A vague opener cannot be clarified retroactively without paying the token cost of resending the entire prior history.

In [ ]:
# ── Multi-turn chat() helper ───────────────────────────────────────────────────
def chat(messages, system="", max_tokens=300, label="chat"):
    """
    Send a multi-turn conversation to Claude and return the reply text.
    Appends the assistant reply to the messages list IN PLACE so the
    caller can keep extending the conversation.

    Parameters
    ----------
    messages   : list[dict] — the growing conversation history
    system     : str        — persistent persona/rules
    max_tokens : int        — output token ceiling
    label      : str        — tag for usage_log

    Returns
    -------
    str — the assistant's reply
    """
    kwargs = dict(
        model="claude-haiku-4-5",
        max_tokens=max_tokens,
        messages=messages          # full history is sent every call — stateless API
    )
    if system:
        kwargs["system"] = system  # system is outside messages list

    response = client.messages.create(**kwargs)
    reply = response.content[0].text   # extract text from first content block

    # Log usage so we can track how input tokens grow with each turn
    usage_log.append({
        "label":          label,
        "prompt_preview": messages[-1]["content"][:60],  # last user message preview
        "input_tokens":   response.usage.input_tokens,
        "output_tokens":  response.usage.output_tokens,
        "stop_reason":    response.stop_reason,
        "max_tokens":     max_tokens
    })

    # Append assistant reply to history so the NEXT call sees the full context
    messages.append({"role": "assistant", "content": reply})
    return reply


# ── Run a 3-turn conversation ─────────────────────────────────────────────────
conversation = []   # start with an empty history

# Turn 1 — must be self-contained; no context exists yet
conversation.append({"role": "user", "content": "What is an API key?"})
t1 = chat(conversation, label="chat-turn-1")
print("Turn 1:", t1[:200], "...")
print()

# Turn 2 — 'it' refers to API key; context from Turn 1 makes this unambiguous
conversation.append({"role": "user", "content": "How should I store it securely?"})
t2 = chat(conversation, label="chat-turn-2")
print("Turn 2:", t2[:200], "...")
print()

# Turn 3 — builds further on the established context
conversation.append({"role": "user", "content": "What happens if it is leaked?"})
t3 = chat(conversation, label="chat-turn-3")
print("Turn 3:", t3[:200], "...")

In [ ]:
# ── Token cost of history accumulation ───────────────────────────────────────
# Extract the three chat turns from the usage log
chat_entries = [e for e in usage_log if e["label"].startswith("chat-turn")]

print(f"{'Turn':<10} {'Label':<16} {'Input Tokens':>14} {'Output Tokens':>14}")
print("-" * 58)
for i, entry in enumerate(chat_entries, 1):
    print(f"{i:<10} {entry['label']:<16} {entry['input_tokens']:>14} {entry['output_tokens']:>14}")

print()
print("NOTE: Each turn's INPUT tokens include ALL prior turns.")
print("      Input token count grows approximately linearly with conversation length.")
print("      This is the cost of stateless history accumulation — the API has no memory;")
print("      the client must resend the full transcript on every call.")

if len(chat_entries) >= 2:
    growth = chat_entries[-1]["input_tokens"] - chat_entries[0]["input_tokens"]
    print(f"\nInput token growth from Turn 1 → Turn 3: +{growth} tokens")

## Section 6: Core Concept — What Makes a Good API Key Explanation?
## CCA objective demonstrated: Operationalising a concept into a deterministic scoring rubric

In [ ]:
import re   # regex module — used for deterministic keyword scoring

# ── Scoring rubric ─────────────────────────────────────────────────────────────
# Each criterion is a function that returns 0 or 1 (int(bool(...))).
# int(bool(x)) converts a truthy/falsy value to 0 or 1:
#   int(bool(None))  → 0
#   int(bool(Match)) → 1
# This lets us sum scores cleanly without if/else branching.

PASS_THRESHOLD = 3   # a response must score >= 3/4 to be considered passing

def score_response(text):
    """
    Evaluate an API key explanation against four deterministic criteria.

    Uses re.search (not re.match) because:
      - re.match only checks the START of the string
      - re.search scans the entire string — correct for multi-sentence replies
    Uses \\b word boundaries to avoid matching substrings:
      - Without \\b: 'authenticate' would match inside 'unauthenticated'
      - With \\b: only exact word/phrase boundaries match

    Parameters
    ----------
    text : str — the assistant reply to evaluate

    Returns
    -------
    dict with keys: defines_api_key, mentions_auth, security_advice, concise, total
    """
    t = text.lower()   # normalise to lowercase so matching is case-insensitive

    # Criterion 1: does the response define what an API key IS?
    defines_api_key = int(bool(
        re.search(r"\bapi key\b", t) or re.search(r"\btoken\b", t)
    ))

    # Criterion 2: does the response mention authentication or authorization?
    mentions_auth = int(bool(
        re.search(r"\bauthentic", t) or re.search(r"\bauthori", t)
    ))

    # Criterion 3: does the response give actionable security advice?
    security_advice = int(bool(
        re.search(r"\bsecret\b", t)
        or re.search(r"\bnever share\b", t)
        or re.search(r"\brotate\b", t)
        or re.search(r"\benviro", t)   # 'environment variable' prefix
    ))

    # Criterion 4: is the response concise (≤ 150 words)?
    word_count = len(text.split())   # str.split() splits on whitespace
    concise = int(word_count <= 150)

    total = defines_api_key + mentions_auth + security_advice + concise

    return {
        "defines_api_key":  defines_api_key,
        "mentions_auth":    mentions_auth,
        "security_advice":  security_advice,
        "concise":          concise,
        "total":            total,
        "word_count":       word_count
    }


def print_scores(scores, label=""):
    """
    Pretty-print a score dict with pass/fail icons per criterion,
    a total, and an overall PASS/FAIL verdict.

    Parameters
    ----------
    scores : dict — output of score_response()
    label  : str  — optional header label
    """
    if label:
        print(f"\n{'='*50}")
        print(f"  {label}")
        print(f"{'='*50}")

    # ✅/❌ icons make pass/fail immediately visible — no need to interpret 0/1
    icon = lambda v: "✅" if v else "❌"   # lambda: True→✅, False/0→❌

    print(f"  Defines API key:   {icon(scores['defines_api_key'])}  {scores['defines_api_key']}/1")
    print(f"  Mentions auth:     {icon(scores['mentions_auth'])}  {scores['mentions_auth']}/1")
    print(f"  Security advice:   {icon(scores['security_advice'])}  {scores['security_advice']}/1")
    print(f"  Concise (≤150w):   {icon(scores['concise'])}  {scores['concise']}/1  [{scores['word_count']} words]")
    print(f"  {'─'*36}")
    print(f"  Total:             {scores['total']}/4")

    # Overall verdict line
    if scores['total'] >= PASS_THRESHOLD:
        print(f"  Verdict:           ✅ PASS (threshold: {PASS_THRESHOLD}/4)")
    else:
        print(f"  Verdict:           ❌ FAIL (threshold: {PASS_THRESHOLD}/4) — iterate")

print("Rubric functions defined. PASS_THRESHOLD =", PASS_THRESHOLD)

## Section 7: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
## CCA objective demonstrated: Applying a structured prompt improvement loop with measurable scoring

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs series (see the CCA eval lab series for
> implementation details).

The cycle has four phases:
1. **Write** — draft a prompt
2. **Evaluate** — score it against the rubric
3. **Improve** — revise the prompt based on which criteria failed
4. **Re-evaluate** — confirm measurable improvement

In [ ]:
# ── Phase 1 & 2: Write a weak prompt and evaluate it ─────────────────────────
WEAK_PROMPT = "Tell me about API keys."
# This prompt is vague: no persona, no length constraint, no required content

v1_reply = ask(WEAK_PROMPT, max_tokens=300, label="v1-weak")
s1 = score_response(v1_reply)

print("V1 reply (first 300 chars):")
print(v1_reply[:300], "...")
print_scores(s1, label="Version 1 — Weak Prompt")

In [ ]:
# ── Phase 3 & 4: Improve the prompt and re-evaluate ───────────────────────────
# Improvements made:
#   - Added security-advocate system prompt (forces mentions_auth + security_advice)
#   - Tightened max_tokens to 150 (forces concise)
#   - Made the user prompt explicit about required content

IMPROVED_PROMPT = (
    "Define what an API key is, explain its role in authentication, "
    "and give one concrete security recommendation. "
    "Keep your answer under 120 words."
)

v2_reply = ask(
    IMPROVED_PROMPT,
    system=SECURITY_SYSTEM,   # stable persona lives in system — not repeated in user turn
    max_tokens=150,            # lower ceiling enforces conciseness
    label="v2-improved"
)
s2 = score_response(v2_reply)

print("V2 reply:")
print(v2_reply)
print_scores(s2, label="Version 2 — Improved Prompt")

# ── Delta ─────────────────────────────────────────────────────────────────────
delta = s2['total'] - s1['total']
print(f"\n>>> Score delta: {s2['total']} - {s1['total']} = {delta:+d}")

In [ ]:
# ── Side-by-side comparison table ────────────────────────────────────────────
# This table makes the improvement measurable at a glance.

def icon(v):
    """Return ✅ for truthy values and ❌ for falsy values."""
    return "✅" if v else "❌"

# Header
print(f"{'Version':<12} {'Prompt summary':<32} {'def':<5} {'auth':<5} {'sec':<5} {'brief':<7} {'Total':<7} {'Result'}")
print("-" * 85)

# Row for V1
r1 = (
    f"{'V1 (weak)':<12} "
    f"{'Tell me about API keys.':<32} "
    f"{icon(s1['defines_api_key']):<5} "
    f"{icon(s1['mentions_auth']):<5} "
    f"{icon(s1['security_advice']):<5} "
    f"{icon(s1['concise']):<7} "
    f"{s1['total']}/4   "
    f"{'PASS' if s1['total'] >= PASS_THRESHOLD else 'FAIL'}"
)
print(r1)

# Row for V2
r2 = (
    f"{'V2 (improved)':<12} "
    f"{'Define + auth + security tip.':<32} "
    f"{icon(s2['defines_api_key']):<5} "
    f"{icon(s2['mentions_auth']):<5} "
    f"{icon(s2['security_advice']):<5} "
    f"{icon(s2['concise']):<7} "
    f"{s2['total']}/4   "
    f"{'PASS' if s2['total'] >= PASS_THRESHOLD else 'FAIL'}"
)
print(r2)
print("-" * 85)
print(f"Score delta: {delta:+d} point(s)")
print()

# ── Pass/fail gate ────────────────────────────────────────────────────────────
if s2['total'] < PASS_THRESHOLD:
    print(f"❌ FAIL — V2 score ({s2['total']}) is below threshold ({PASS_THRESHOLD}). Iterate again.")
else:
    print(f"✅ PASS — V2 score ({s2['total']}) meets or exceeds threshold ({PASS_THRESHOLD}).")

## Section 8: Failure Mode Analysis
## CCA objective demonstrated: Identifying, cataloguing, and live-demonstrating common API failure modes

| # | Failure Mode | Root Cause | Symptom | Fix |
|---|---|---|---|---|
| 1 | Missing `max_tokens` | Required param omitted | `BadRequestError` at call time | Always pass `max_tokens=` |
| 2 | Expired / invalid key | Wrong key in `.env` | `AuthenticationError` (401) | Regenerate key in console |
| 3 | Truncated reply | `max_tokens` too low | `stop_reason == 'max_tokens'` | Increase `max_tokens` |
| 4 | Vague prompt → high variance | No constraints in prompt | Output length and format vary wildly across runs | Add explicit length + format instructions |
| 5 | System prompt ignored | Conflicting user instruction | Model follows user, not system | Reinforce rules in both; use `prefill` |
| 6 | Token budget blowout | History grows unbounded | Cost spikes in long sessions | Summarise history periodically |
| 7 | `temperature=1` in classifier | High randomness | Non-deterministic labels | Use `temperature=0` for classifiers |
| 8 | `re.match` misses mid-text keywords | Wrong regex method | Eval under-scores correct replies | Use `re.search` for full-text scan |

In [ ]:
# ── Live demo: output variance on a vague prompt ─────────────────────────────
# Failure mode #4: a vague prompt with no constraints produces non-deterministic
# output length and format — a production bug, not an aesthetic preference.
#
# We run the SAME vague prompt 3 times at temperature=1 (max randomness)
# and measure: word count per run, word count range, and format consistency.

VAGUE_PROMPT = "Tell me about API keys."   # no length, format, or content constraints
N_RUNS = 3
runs = []

for i in range(N_RUNS):
    reply = ask(
        VAGUE_PROMPT,
        max_tokens=400,
        temperature=1,           # max randomness — shows variance clearly
        label=f"variance-run-{i+1}"
    )
    word_count = len(reply.split())                      # count whitespace-delimited words
    uses_bullets = bool(re.search(r"^\s*[-•*]", reply, re.MULTILINE))  # detect bullet lists
    runs.append({
        "run": i + 1,
        "word_count": word_count,
        "uses_bullets": uses_bullets,
        "preview": reply[:80].replace("\n", " ")
    })

# ── Print results table ───────────────────────────────────────────────────────
print(f"Vague prompt run {N_RUNS}× at temperature=1")
print(f"Prompt: '{VAGUE_PROMPT}'")
print()
print(f"{'Run':<6} {'Words':<8} {'Bullets?':<10} {'Preview (80 chars)'}")
print("-" * 70)
for r in runs:
    bullet_str = "yes" if r["uses_bullets"] else "no"
    print(f"{r['run']:<6} {r['word_count']:<8} {bullet_str:<10} {r['preview']}")

word_counts   = [r["word_count"] for r in runs]    # list comprehension: extract word counts
wc_range      = max(word_counts) - min(word_counts) # range = max - min
bullet_runs   = sum(1 for r in runs if r["uses_bullets"])  # count runs that used bullets

print()
print(f"Word count range (max − min): {max(word_counts)} − {min(word_counts)} = {wc_range}")
print(f"Format consistency: {bullet_runs}/{N_RUNS} runs used bullet points")
print()
print("⚠  Non-deterministic output length is a PRODUCTION BUG, not an aesthetic preference.")
print("   Downstream parsers, UI layouts, and token budgets all break when output size varies.")
print("   Fix: add explicit length and format constraints to the prompt.")

## Section 9: Token Usage Analysis
## CCA objective demonstrated: Auditing session-wide token consumption and interpreting cost signals

In [ ]:
# ── Full session audit ────────────────────────────────────────────────────────
print(f"{'#':<4} {'Label':<20} {'In':>8} {'Out':>8} {'Total':>8} {'Stop':<12} {'MaxTok':>8}")
print("-" * 75)

session_in = session_out = 0
for i, entry in enumerate(usage_log, 1):
    total_tok = entry["input_tokens"] + entry["output_tokens"]
    session_in  += entry["input_tokens"]
    session_out += entry["output_tokens"]
    # Flag rows where stop_reason is max_tokens — possible truncation
    trunc_flag = " ⚠" if entry["stop_reason"] == "max_tokens" else ""
    print(
        f"{i:<4} {entry['label']:<20} "
        f"{entry['input_tokens']:>8} {entry['output_tokens']:>8} {total_tok:>8} "
        f"{entry['stop_reason'] + trunc_flag:<12} {entry['max_tokens']:>8}"
    )

print("-" * 75)
print(f"{'SESSION TOTALS':<24} {session_in:>8} {session_out:>8} {session_in+session_out:>8}")
print()
print("⚠ rows indicate stop_reason='max_tokens' — the reply may be truncated.")

In [ ]:
# ── Output token variance: weak prompt vs. improved prompt ────────────────────
# The improved prompt uses a tighter max_tokens cap AND explicit length instructions.
# We expect it to produce fewer output tokens — lower cost per call.

v1_entry = next((e for e in usage_log if e["label"] == "v1-weak"), None)
v2_entry = next((e for e in usage_log if e["label"] == "v2-improved"), None)

if v1_entry and v2_entry:
    print("Output token comparison: weak prompt vs. improved prompt")
    print(f"{'Version':<20} {'label':<16} {'max_tokens':>12} {'output_tokens':>14}")
    print("-" * 65)
    print(f"{'V1 (weak)':<20} {v1_entry['label']:<16} {v1_entry['max_tokens']:>12} {v1_entry['output_tokens']:>14}")
    print(f"{'V2 (improved)':<20} {v2_entry['label']:<16} {v2_entry['max_tokens']:>12} {v2_entry['output_tokens']:>14}")
    print("-" * 65)
    saved = v1_entry['output_tokens'] - v2_entry['output_tokens']
    print(f"Output token reduction: {saved:+d} tokens")
    print()
    if saved > 0:
        print("✅ The improved prompt produced fewer output tokens.")
        print("   Tighter max_tokens + explicit length instructions = lower cost per call.")
    elif saved == 0:
        print("➡  Both prompts produced the same output token count.")
    else:
        print("⚠  The improved prompt used MORE output tokens — review the length instruction.")
else:
    print("Could not find v1 or v2 log entries — run Section 7 cells first.")

## Section 10: Practice Drills
## CCA objective demonstrated: Independently applying API key, system prompt, and rubric patterns

In [ ]:
# ── Drill 1: Custom system prompt ─────────────────────────────────────────────
# Task: Write a system prompt that makes Claude respond like a pirate,
# then ask it to explain API keys. Verify the persona is applied.

pirate_system = "You are a pirate. Speak in pirate dialect. Keep answers under 80 words."

pirate_reply = ask(
    "What be this 'API key' ye speak of?",
    system=pirate_system,
    max_tokens=120,
    label="drill-1-pirate"
)
print("Drill 1 — Pirate persona:")
print(pirate_reply)
print()

In [ ]:
# ── Drill 2: Extend the rubric ─────────────────────────────────────────────────
# Task: Add a 5th criterion — does the response mention 'environment variable'?
# Extend score_response() and test it on the v2 reply.

def score_response_v2(text):
    """
    Extended rubric with a 5th criterion: mentions environment variable storage.
    Inherits all four criteria from score_response() and adds one more.
    """
    base = score_response(text)   # run the original four criteria
    t = text.lower()

    # Criterion 5: does the response mention environment variables as a storage method?
    mentions_env_var = int(bool(
        re.search(r"\benv(ironment)?\b", t) or re.search(r"\.env\b", t)
    ))

    base["mentions_env_var"] = mentions_env_var
    base["total"] += mentions_env_var   # add to existing total
    return base

s2_extended = score_response_v2(v2_reply)
print("Drill 2 — Extended rubric on V2 reply:")
print(f"  Mentions env var: {'✅' if s2_extended['mentions_env_var'] else '❌'}  {s2_extended['mentions_env_var']}/1")
print(f"  Extended total:   {s2_extended['total']}/5")

In [ ]:
# ── Drill 3: Multi-turn with a system prompt ───────────────────────────────────
# Task: Start a 2-turn conversation with a system prompt that constrains
# the assistant to only discuss security topics. Ask an off-topic question
# in Turn 2 and observe how the system prompt shapes the refusal.

security_only_system = (
    "You are a security specialist. You ONLY discuss topics related to "
    "cybersecurity, authentication, and API security. "
    "If asked about anything else, politely redirect to security topics."
)

drill3_msgs = []
drill3_msgs.append({"role": "user", "content": "How do I keep my API key safe?"})
r1 = chat(drill3_msgs, system=security_only_system, max_tokens=100, label="drill-3-turn-1")
print("Drill 3, Turn 1:", r1[:150])
print()

# Off-topic question — the system prompt should redirect
drill3_msgs.append({"role": "user", "content": "What is the best recipe for chocolate cake?"})
r2 = chat(drill3_msgs, system=security_only_system, max_tokens=100, label="drill-3-turn-2")
print("Drill 3, Turn 2 (off-topic — observe redirect):", r2[:200])

## Section 11: CCA Takeaways
## CCA objective demonstrated: Consolidating exam-ready mental models for API key and authentication concepts

| # | Mental Model | Why It Matters |
|---|---|---|
| 1 | **`max_tokens` is required** — omitting it raises `BadRequestError` immediately | Guards against unbounded generation and enforces budget discipline |
| 2 | **`system` ≠ `messages[0]`** — system is a top-level keyword arg, not a message turn | Keeps stable instructions separate from per-request content; processed differently by the API |
| 3 | **The API is stateless** — full history must be resent every call; input tokens grow linearly | Designing for token budget means designing for history management from Turn 1 |
| 4 | **Deterministic rubrics over-score on keywords; use Claude-as-judge for semantics** | Keyword match on 'authenticate' succeeds even if the sentence says 'do not authenticate' |
| 5 | **Vague prompts are non-deterministic at `temperature > 0`** — output length and format vary across runs | Production systems require explicit length and format constraints; variance is a bug, not a feature |